In this notebook I want to compute phase-slope index. 

In [ ]:
import numpy as np
import mne
import mne_connectivity
import tensorpac
from tensorpac import Pac
from tensorpac.signals import pac_signals_wavelet
from mne_connectivity import phase_slope_index, seed_target_indices, spectral_connectivity_epochs
from mne.stats import permutation_cluster_test, permutation_cluster_1samp_test
from mne.filter import next_fast_len
from mne.preprocessing.bads import _find_outliers

from glob import glob
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
import seaborn as sns
from scipy.stats import zscore, linregress, ttest_ind, ttest_rel, ttest_1samp, pearsonr
import pandas as pd
from scipy.signal import hilbert, correlate, correlation_lags
import fooof
# Import plotting functions
from fooof.plts.spectra import plot_spectrum, plot_spectra
# Import the FOOOF object
from fooof import FOOOF
from fooof import FOOOFGroup
import os 
import joblib
import statsmodels.formula.api as smf
import statsmodels.api as sm
import pickle
from tqdm import tqdm
from IPython.display import clear_output
import warnings 
from ast import literal_eval


# I only want to see warnings once
warnings.filterwarnings('ignore')

from joblib import delayed, Parallel
from statsmodels.stats import multitest

plot_dir = '/Users/salmanqasim/Documents/Projects/MemBanditsEMU/Plots/MemoryBandit-PSI'
os.makedirs(plot_dir, exist_ok=True)

In [ ]:
import sys
sys.path.append('/Users/salmanqasim/Documents/GitRepos/LFPAnalysis')


In [ ]:
from LFPAnalysis import lfp_preprocess_utils, sync_utils, analysis_utils, nlx_utils, oscillation_utils, statistics_utils


In [ ]:
# Setup paths
base_dir = '/Volumes/T7 Shield'
load_dir = f'{base_dir}/work/qasims01/MemoryBanditData/EMU'

# Load demographics data
demo_file = f'{base_dir}/projects/guLab/Salman/EMU/MemoryBanditsDatabase.xlsx'
demo_data = pd.read_excel(demo_file, engine='openpyxl')
subj_df = demo_data[(demo_data.LongVersion==1) & (demo_data.Minerva==1)]

# Filter out Iowa patients
subj_df = subj_df[~subj_df['MSSMCode'].isin(['MS015', 'MS024', 'MS034', 'MS038'])]

print(f"Available subjects: {subj_df.MSSMCode.values}")

In [ ]:
# load behavioral data
learn_df = pd.read_csv(f'{load_dir}/learn_df_RWH.csv')
combined_df = pd.read_csv(f'{load_dir}/full_df_RWH.csv')

In [ ]:
# Explicitly define a list of analysis events and the baseline event. Should correspond to the dict

evs = {'gamble_start':[-1.0, 0], 
       'feedback_start': [0, 1.5],
       'baseline_start': [0, 0.75]}

baseline_ev = 'baseline_start'
analysis_evs = ['feedback_start']

day = 'Day1'

for i in analysis_evs: 
    if i not in evs.keys(): 
        raise ValueError('This analysis event is not in the dictionary')

In [ ]:
metric = 'psi'
freqs = np.logspace(*np.log10([2, 200]), num=30)
n_cycles = np.floor(np.logspace(*np.log10([3, 10]), num=30))

freq_dict = {'theta':[2, 8],
            'alpha':[8, 13],
            'beta': [13, 30], 
            'hfa': [70, 200]}

band1 = 'theta'
band2 = 'theta'

buf_ms = 1000
n_surr = 500

savepath = f'/sc/arion/projects/guLab/Salman/EphysAnalyses/Connectivity/scratch/{metric}'
if not os.path.exists(savepath):
    os.makedirs(savepath)

Run the PSI in two conditions:

In [ ]:
%%capture

# pairs = (['AMY', 'HPC'], ['AMY', 'OFC'], ['AMY', 'dmPFC'], ['AMY', 'ACC'])
pairs = (['HPC', 'dmPFC'], )

for pair in pairs:
    source_region = pair[0]
    target_region = pair[1]
    
    # initialize the storage list 
    conn_group_data = []
    conn_group_metadata = []
    
    # iterate through subjects
    for subj_id in subj_ids:
        for event in analysis_evs:
            loadpath = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}'
            epochs_reref = mne.read_epochs(f'{loadpath}/neural/{day}/{event}-epo.fif', preload=True); 
            
            pos_rpe_index = np.where(learn_df[learn_df.participant==subj_id].rpe>0)[0]           

            # Get electrode df 
            elec_df = pd.read_csv(f'{loadpath}/Day1_reref_elec_df')

            # construct the seed-to-target mapping based on your rois - matters most for PSI as this is directional 
            seed_target_df = pd.DataFrame(columns=['seed', 'target'], index=['l', 'r'])
            for hemi in ['l', 'r']:
                seed_target_df['seed'][hemi] = np.where(elec_df.loc[elec_df.hemisphere==hemi, 'salman_region'] == source_region)[0]
#                 seed_target_df['target'][hemi] = np.where(elec_df.loc[elec_df.hemisphere==hemi, 'salman_region'].isin(target_regions))[0]    

                seed_target_df['target'][hemi] = np.where(elec_df.loc[elec_df.hemisphere==hemi, 'salman_region'] == target_region)[0]    

            seed_target_df = seed_target_df[
                        (seed_target_df['seed'].map(lambda d: len(d) > 0)) & (seed_target_df['target'].map(lambda d: len(d) > 0))]

            if len(seed_target_df) == 0:
                continue
                
            # for cond in conditions: 
            for hemi in ['l', 'r']:
                # first determine if ipsilateral connectivity is even possible; if not, move on. 
                # note: edit this if you want to compute contralateral connectivity 
                if hemi not in seed_target_df.index.tolist():
                    continue
                else:
                    seed_to_target = seed_target_indices(
                        seed_target_df['seed'][hemi],
                        seed_target_df['target'][hemi])
            
            
                pwise = oscillation_utils.compute_connectivity(epochs_reref.copy()[pos_rpe_index], 
                                                   band = freq_dict[band1], 
                                                   metric = metric, 
                                                   indices = seed_to_target, 
                                                   freqs = freqs, 
                                                   n_cycles = n_cycles,
                                                   buf_ms = buf_ms, 
                                                   n_surr=n_surr,
                                                   avg_over_dim='epochs',
                                                   band1 = freq_dict[band2],
                                                   parallelize=True);

                npairs = len(seed_to_target[0])
                pair_labels = [] 
                for pair_ix in range(npairs):
                    pair_labels.append(f'{elec_df[elec_df.hemisphere==hemi].iloc[seed_to_target[0][pair_ix]].label}:{elec_df[elec_df.hemisphere==hemi].iloc[seed_to_target[1][pair_ix]].label}')


                row_metadata = pd.DataFrame({
                    'pair_labels': pair_labels,
                    'participant': subj_id,
                    'age': subj_df[subj_df.MSSMCode==subj_id].Age.values[0], 
                    'sex': subj_df[subj_df.MSSMCode==subj_id].Sex.str.strip().values[0],
                    'roi1': source_region, 
                    'roi2': target_region, 
                    'hemi': hemi, 
                    'metric': metric,
                    'event': event,
                    'band0': band1, 
                    'band1': band2,
                    'cond':'prpe'})

                conn_group_data.append(pwise)
                conn_group_metadata.append(row_metadata)
            
    conn_group_data = np.vstack(conn_group_data)
    conn_group_metadata = pd.concat(conn_group_metadata)
    
    np.save(f'{savepath}/{source_region}_{target_region}_{metric}_{band1}_{band2}_{event}_pos_rpe.npy', conn_group_data)
    conn_group_metadata.to_csv(f'{savepath}/{source_region}_{target_region}_{metric}_{band1}_{band2}_{event}_pos_rpe_df', index=False)
    

In [ ]:
%%capture

pairs = (['HPC', 'dmPFC'], )

for pair in pairs:
    source_region = pair[0]
    target_region = pair[1]
    
    # initialize the storage list 
    conn_group_data = []
    conn_group_metadata = []
    
    # iterate through subjects
    for subj_id in subj_ids:
        for event in analysis_evs:
            loadpath = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}'
            epochs_reref = mne.read_epochs(f'{loadpath}/neural/{day}/{event}-epo.fif', preload=True); 
            
            neg_rpe_index = np.where(learn_df[learn_df.participant==subj_id].rpe<0)[0]

            # Get electrode df 
            elec_df = pd.read_csv(f'{loadpath}/Day1_reref_elec_df')

            # construct the seed-to-target mapping based on your rois - matters most for PSI as this is directional 
            seed_target_df = pd.DataFrame(columns=['seed', 'target'], index=['l', 'r'])
            for hemi in ['l', 'r']:
                seed_target_df['seed'][hemi] = np.where(elec_df.loc[elec_df.hemisphere==hemi, 'salman_region'] == source_region)[0]
#                 seed_target_df['target'][hemi] = np.where(elec_df.loc[elec_df.hemisphere==hemi, 'salman_region'].isin(target_regions))[0]    

                seed_target_df['target'][hemi] = np.where(elec_df.loc[elec_df.hemisphere==hemi, 'salman_region'] == target_region)[0]    

            seed_target_df = seed_target_df[
                        (seed_target_df['seed'].map(lambda d: len(d) > 0)) & (seed_target_df['target'].map(lambda d: len(d) > 0))]

            if len(seed_target_df) == 0:
                continue
                
            # for cond in conditions: 
            for hemi in ['l', 'r']:
                # first determine if ipsilateral connectivity is even possible; if not, move on. 
                # note: edit this if you want to compute contralateral connectivity 
                if hemi not in seed_target_df.index.tolist():
                    continue
                else:
                    seed_to_target = seed_target_indices(
                        seed_target_df['seed'][hemi],
                        seed_target_df['target'][hemi])
            
            
                pwise = oscillation_utils.compute_connectivity(epochs_reref.copy()[neg_rpe_index], 
                                                   band = freq_dict[band1], 
                                                   metric = metric, 
                                                   indices = seed_to_target, 
                                                   freqs = freqs, 
                                                   n_cycles = n_cycles,
                                                   buf_ms = buf_ms, 
                                                   n_surr=n_surr,
                                                   avg_over_dim='epochs',
                                                   band1 = freq_dict[band2],
                                                   parallelize=True);

                npairs = len(seed_to_target[0])
                pair_labels = [] 
                for pair_ix in range(npairs):
                    pair_labels.append(f'{elec_df[elec_df.hemisphere==hemi].iloc[seed_to_target[0][pair_ix]].label}:{elec_df[elec_df.hemisphere==hemi].iloc[seed_to_target[1][pair_ix]].label}')


                row_metadata = pd.DataFrame({
                    'pair_labels': pair_labels,
                    'participant': subj_id,
                    'age': subj_df[subj_df.MSSMCode==subj_id].Age.values[0], 
                    'sex': subj_df[subj_df.MSSMCode==subj_id].Sex.str.strip().values[0],
                    'roi1': source_region, 
                    'roi2': target_region, 
                    'hemi': hemi, 
                    'metric': metric,
                    'event': event,
                    'band0': band1, 
                    'band1': band2,
                    'cond':'nrpe'})

                conn_group_data.append(pwise)
                conn_group_metadata.append(row_metadata)
            
    conn_group_data = np.vstack(conn_group_data)
    conn_group_metadata = pd.concat(conn_group_metadata)
    
    np.save(f'{savepath}/{source_region}_{target_region}_{metric}_{band1}_{band2}_{event}_neg_rpe.npy', conn_group_data)
    conn_group_metadata.to_csv(f'{savepath}/{source_region}_{target_region}_{metric}_{band1}_{band2}_{event}_neg_rpe_df', index=False)
    

In [ ]:
source_region = 'OFC'
target_region = 'HPC'
band1= 'theta'
band2= 'theta'
event = 'feedback_start'
fig, ax = plt.subplots(1, 1, figsize=(5, 3), dpi=300)
for target_region in ['HPC']:
    conn_group_data_pos = np.load(f'{savepath}/{source_region}_{target_region}_{metric}_{band1}_{band2}_{event}_pos_rpe_Z.npy')
    conn_group_data_neg = np.load(f'{savepath}/{source_region}_{target_region}_{metric}_{band1}_{band2}_{event}_neg_rpe_Z.npy')

    T_obs, clusters, cluster_p_values, H0 = mne.stats.permutation_cluster_test([conn_group_data_pos, conn_group_data_neg], 
                                                                                     n_permutations=1000,
                                                                                     out_type='mask')
    print(cluster_p_values)
    stderr = np.nanstd(conn_group_data_pos, axis=0) / np.sqrt(conn_group_data_pos.shape[0])
    mean_data = np.nanmean(conn_group_data_pos, axis=0)
    
    times = np.linspace(0, 1.5, conn_group_data_pos.shape[1])
    
    ax.plot(times, mean_data)
    ax.fill_between(times, mean_data - stderr, mean_data + stderr, color='green', alpha=0.3)
    
    stderr = np.nanstd(conn_group_data_neg, axis=0) / np.sqrt(conn_group_data_neg.shape[0])
    mean_data = np.nanmean(conn_group_data_neg, axis=0)
    
    times = np.linspace(0, 1.5, conn_group_data_neg.shape[1])
    
    ax.plot(times, mean_data)
    ax.fill_between(times, mean_data - stderr, mean_data + stderr, color='red', alpha=0.3)

    ax.hlines(0, 0, 1.5, 'k')
    
    for i_c, c in enumerate(clusters):
        c = c[0]
        if cluster_p_values[i_c] <= 0.05:
            if target_region=='dmPFC':
                h = ax.axvspan(times[c.start], times[c.stop - 1], color="red", alpha=0.3)
            elif target_region=='OFC':
                h = ax.axvspan(times[c.start], times[c.stop - 1], color="yellow", alpha=0.3)
            else: 
                h = ax.axvspan(times[c.start], times[c.stop - 1], color="k", alpha=0.3)
#         else:
#             ax.axvspan(times[c.start], times[c.stop - 1], color=(0.3, 0.3, 0.3), alpha=0.3)


sns.despine()
ax.set(xlabel='time (s)', ylabel='theta PSI')
plt.savefig(f'{plot_dir}/PSI_{source_region}_{target_region}_{band1}_Z.pdf', dpi=300)
